# Week 7 – Fine-Tuning an Open-Source Model for Product Pricing

Fine-tune **Llama 3.2 3B** with QLoRA to predict product prices from descriptions.

| Step | What we do |
|------|------------|
| 1 | Install dependencies & authenticate |
| 2 | Load the prompt dataset from HuggingFace |
| 3 | Load tokenizer & base model (4-bit quantized) |
| 4 | Test the base model before fine-tuning |
| 5 | Configure LoRA & train with SFTTrainer |
| 6 | Push the fine-tuned adapter to HuggingFace Hub |
| 7 | Load the fine-tuned model & evaluate |

**Requirements:** Google Colab with a T4 GPU (free tier works), HuggingFace token with write access.

In [ ]:
%pip install -q datasets torch transformers bitsandbytes accelerate peft trl matplotlib plotly scikit-learn

In [ ]:
import os
import re
import math
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from datetime import datetime
from tqdm import tqdm
from huggingface_hub import login
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    set_seed,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv(override=True)
    hf_token = os.environ["HF_TOKEN"]

login(hf_token, add_to_git_credential=True)

%matplotlib inline
print(f"PyTorch {torch.__version__}  CUDA available: {torch.cuda.is_available()}")

## Configuration

Change `HF_USER` to your own HuggingFace username so the fine-tuned adapter is pushed to your account.

In [ ]:
BASE_MODEL = "meta-llama/Llama-3.2-3B"
HF_USER = "williampepple1"  
DATASET_NAME = "ed-donner/pricer-data"

QUANT_4_BIT = True

GREEN = "\033[92m"
YELLOW = "\033[93m"
RED = "\033[91m"
RESET = "\033[0m"
COLOR_MAP = {"red": RED, "orange": YELLOW, "green": GREEN}

## 1 – Load the dataset

In [ ]:
dataset = load_dataset(DATASET_NAME)
train_ds = dataset["train"]
test_ds = dataset["test"]

print(f"Train : {len(train_ds):,}")
print(f"Test  : {len(test_ds):,}")
print(f"\nSample:\n{test_ds[0]}")

In [ ]:
prices = [item["price"] for item in train_ds]

fig, ax = plt.subplots(1, 1, figsize=(14, 5))
ax.hist(prices, bins=range(0, 1000, 10), color="blueviolet", rwidth=0.7)
ax.set_title(f"Training price distribution  (avg ${sum(prices)/len(prices):,.1f})")
ax.set_xlabel("Price ($)")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## 2 – Load tokenizer & base model (quantized)

In [ ]:
if QUANT_4_BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
    )
else:
    quant_config = BitsAndBytesConfig(
        load_in_8bit=True,
        bnb_8bit_compute_dtype=torch.bfloat16,
    )

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Model loaded  Memory: {base_model.get_memory_footprint() / 1e6:.0f} MB")

## 3 – Prediction helpers

In [ ]:
def extract_price(text):
    """Pull a numeric price from model output."""
    if "Price is $" in text:
        after = text.split("Price is $")[1].replace(",", "")
        match = re.search(r"[-+]?\d*\.\d+|\d+", after)
        return float(match.group()) if match else 0.0
    return 0.0


def simple_predict(model, prompt, device="cuda"):
    """Generate a price by decoding the next few tokens."""
    set_seed(42)
    inputs = tokenizer.encode(prompt, return_tensors="pt").to(device)
    attention_mask = torch.ones(inputs.shape, device=device)
    outputs = model.generate(
        inputs,
        attention_mask=attention_mask,
        max_new_tokens=5,
        temperature=0.1,
        num_return_sequences=1,
    )
    return extract_price(tokenizer.decode(outputs[0]))


TOP_K = 5

def weighted_predict(model, prompt, device="cuda"):
    """Weighted average of top-K next-token price predictions."""
    set_seed(42)
    inputs = tokenizer.encode(prompt, return_tensors="pt").to(device)
    attention_mask = torch.ones(inputs.shape, device=device)

    with torch.no_grad():
        logits = model(inputs, attention_mask=attention_mask).logits[:, -1, :].cpu()

    probs = F.softmax(logits, dim=-1)
    top_probs, top_ids = probs.topk(TOP_K)

    prices, weights = [], []
    for i in range(TOP_K):
        token_text = tokenizer.decode(top_ids[0][i])
        try:
            value = float(token_text)
        except ValueError:
            value = 0.0
        if value > 0:
            prices.append(value)
            weights.append(top_probs[0][i])

    if not prices:
        return 0.0
    total_w = sum(weights)
    return sum(p * w / total_w for p, w in zip(prices, weights)).item()


print("Prediction helpers ready.")

## 4 – Evaluate the base model (before fine-tuning)

In [ ]:
def color_for(error, truth):
    if error < 40 or error / truth < 0.2:
        return "green"
    elif error < 80 or error / truth < 0.4:
        return "orange"
    return "red"


def evaluate_model(model, data, title, size=200):
    """Run predictions on `size` test items and report results."""
    errors = []
    guesses = []
    truths = []
    colors = []

    for i in tqdm(range(size)):
        prompt = data[i]["text"]
        truth = data[i]["price"]
        guess = weighted_predict(model, prompt)
        error = abs(guess - truth)
        c = color_for(error, truth)
        errors.append(error)
        guesses.append(guess)
        truths.append(truth)
        colors.append(c)
        print(f"{COLOR_MAP[c]}{i+1}: Guess=${guess:,.2f}  Truth=${truth:,.2f}  Error=${error:,.2f}{RESET}")

    avg_error = sum(errors) / len(errors)
    rmsle = math.sqrt(sum((math.log(t+1) - math.log(g+1))**2 for t, g in zip(truths, guesses)) / len(truths))
    hits = sum(1 for c in colors if c == "green")

    print(f"\n{title}")
    print(f"  Avg error : ${avg_error:,.2f}")
    print(f"  RMSLE     : {rmsle:,.4f}")
    print(f"  Hits      : {hits}/{size} ({hits/size*100:.1f}%)")

    max_val = max(max(truths), max(guesses))
    plt.figure(figsize=(10, 8))
    plt.scatter(truths, guesses, c=colors, s=4)
    plt.plot([0, max_val], [0, max_val], "--", color="deepskyblue", lw=2)
    plt.xlabel("Actual Price")
    plt.ylabel("Predicted Price")
    plt.xlim(0, max_val)
    plt.ylim(0, max_val)
    plt.title(f"{title}\nAvg Error=${avg_error:,.2f}  RMSLE={rmsle:,.4f}  Hits={hits/size*100:.1f}%")
    plt.show()

    return avg_error, rmsle

In [ ]:
base_error, base_rmsle = evaluate_model(
    base_model, test_ds, "Base Llama 3.2 3B (4-bit)", size=200
)

## 5 – Configure LoRA & train with SFTTrainer

We use QLoRA: freeze the base model, attach small trainable adapters to the attention layers.

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=64,
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

run_name = datetime.now().strftime("%Y-%m-%d_%H.%M.%S")
project_run = f"{PROJECT_NAME}-{run_name}"

training_args = TrainingArguments(
    output_dir=project_run,
    run_name=project_run,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=25,
    save_steps=500,
    save_total_limit=2,
    fp16=True,
    report_to="none",
    weight_decay=0.01,
    seed=42,
)

print(f"Run name: {project_run}")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")

In [ ]:
trainer = SFTTrainer(
    model=base_model,
    train_dataset=train_ds,
    args=training_args,
    peft_config=lora_config,
)

print(f"Trainable parameters: {sum(p.numel() for p in trainer.model.parameters() if p.requires_grad):,}")
print(f"Total parameters:     {sum(p.numel() for p in trainer.model.parameters()):,}")

In [ ]:
trainer.train()

## 6 – Push the fine-tuned adapter to HuggingFace Hub

In [ ]:
trainer.model.push_to_hub(f"{HF_USER}/{project_run}")
tokenizer.push_to_hub(f"{HF_USER}/{project_run}")

print(f"Adapter pushed to: https://huggingface.co/{HF_USER}/{project_run}")

## 7 – Evaluate the fine-tuned model

We reload the base model and apply the LoRA adapter on top.

In [ ]:
fine_tuned_model = PeftModel.from_pretrained(
    base_model,
    f"{HF_USER}/{project_run}",
)

print(f"Fine-tuned model loaded  Memory: {fine_tuned_model.get_memory_footprint() / 1e6:.0f} MB")

In [ ]:
ft_error, ft_rmsle = evaluate_model(
    fine_tuned_model, test_ds, "Fine-tuned Llama 3.2 3B (QLoRA)", size=200
)

## 8 – Comparison

In [ ]:
print("=" * 55)
print(f"{'Model':<35} {'Avg Error':>10} {'RMSLE':>8}")
print("-" * 55)
print(f"{'Base Llama 3.2 3B (4-bit)':<35} ${base_error:>9,.2f} {base_rmsle:>8.4f}")
print(f"{'Fine-tuned Llama 3.2 3B (QLoRA)':<35} ${ft_error:>9,.2f} {ft_rmsle:>8.4f}")
improvement = (base_error - ft_error) / base_error * 100
print("-" * 55)
print(f"Improvement: {improvement:.1f}%")
print("=" * 55)

labels = ["Base Llama 3.2 3B", "Fine-tuned (QLoRA)"]
vals = [base_error, ft_error]
bar_colors = ["darkred", "green"]

plt.figure(figsize=(8, 5))
bars = plt.bar(labels, vals, color=bar_colors, width=0.5)
for bar, v in zip(bars, vals):
    plt.text(bar.get_x() + bar.get_width() / 2, v + 1, f"${v:,.2f}", ha="center", fontweight="bold")
plt.ylabel("Average Absolute Error ($)")
plt.title("Base vs Fine-tuned Model")
plt.tight_layout()
plt.show()